In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.silver.bank_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. READ SILVER SOURCE TABLE
# ============================================================

source_df = spark.read.table("fintech360.silver.bank_data")

# ============================================================
# 2. PREPARE COLUMNS FOR VALIDATION
# Preserve original source columns in the final output.
# ============================================================

df = (source_df
    .withColumn("_bank_txn_id", F.trim(F.col("bank_txn_id")))
    .withColumn("_customer_id", F.col("customer_id").cast("long"))
    .withColumn("_account_id", F.trim(F.col("account_id")))
    .withColumn("_transaction_ts", F.col("transaction_ts").cast("long"))
    .withColumn("_transaction_type", F.upper(F.trim(F.col("transaction_type"))))
    .withColumn("_transaction_amount", F.col("transaction_amount").cast("double"))
    .withColumn("_signed_amount", F.col("signed_amount").cast("double"))
    .withColumn("_balance", F.col("balance_after_txn").cast("double"))
    .withColumn("_currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("_bank_name", F.upper(F.trim(F.col("bank_name"))))
    .withColumn("_salary_flag", F.col("is_salary_like").cast("int"))
    .withColumn("_source_file_id", F.trim(F.col("source_file_id")))
    .withColumn("_created_at", F.col("created_at").cast("long"))
    .withColumn("_updated_at", F.col("updated_at").cast("long"))
)

# ============================================================
# 3. CONVERT UNIX SECONDS TO TIMESTAMPS
# ============================================================

for c in ["transaction_ts", "created_at", "updated_at"]:
    df = df.withColumn(
        f"_dt_{c}",
        F.to_timestamp(F.from_unixtime(F.col(f"_{c}")))
    )

# ============================================================
# 4. IDENTIFY DUPLICATE TRANSACTION IDS
# Quarantine every row sharing the same non-empty ID.
# ============================================================

txn_window = Window.partitionBy("_bank_txn_id")

df = df.withColumn(
    "_txn_id_count",
    F.when(
        F.col("_bank_txn_id").isNotNull() &
        (F.col("_bank_txn_id") != ""),
        F.count("*").over(txn_window)
    ).otherwise(F.lit(0))
)

# ============================================================
# 5. DEFINE DATA QUALITY RULES
# ============================================================

allowed_banks = [
    "SYNTHETIC_BANK_A",
    "SYNTHETIC_BANK_B",
    "SYNTHETIC_BANK_C",
    "SYNTHETIC_BANK_D"
]

allowed_transaction_types = ["CREDIT", "DEBIT"]
allowed_currencies = ["INR"]
allowed_salary_flags = [0, 1]

rules = [
    # Transaction identifier
    (
        F.col("bank_txn_id").isNull() |
        (F.trim(F.col("bank_txn_id")) == ""),
        "MISSING_BANK_TXN_ID"
    ),
    (
        F.col("_txn_id_count") > 1,
        "DUPLICATE_BANK_TXN_ID"
    ),

    # Customer and account
    (
        F.col("customer_id").isNull() |
        F.col("_customer_id").isNull() |
        (F.col("_customer_id") <= 0),
        "INVALID_CUSTOMER_ID"
    ),
    (
        F.col("account_id").isNull() |
        (F.trim(F.col("account_id")) == ""),
        "MISSING_ACCOUNT_ID"
    ),

    # Bank name
    (
        F.col("_bank_name").isNull() |
        (~F.col("_bank_name").isin(allowed_banks)),
        "INVALID_BANK_NAME"
    ),

    # Transaction type
    (
        F.col("_transaction_type").isNull() |
        (~F.col("_transaction_type").isin(allowed_transaction_types)),
        "INVALID_TRANSACTION_TYPE"
    ),

    # Transaction amount
    (
        F.col("transaction_amount").isNull() |
        F.col("_transaction_amount").isNull() |
        F.isnan(F.col("_transaction_amount")) |
        (F.abs(F.col("_transaction_amount")) == float("inf")) |
        (F.col("_transaction_amount") <= 0),
        "INVALID_TRANSACTION_AMOUNT"
    ),

    # Signed amount
    (
        F.col("signed_amount").isNull() |
        F.col("_signed_amount").isNull() |
        F.isnan(F.col("_signed_amount")) |
        (F.abs(F.col("_signed_amount")) == float("inf")),
        "INVALID_SIGNED_AMOUNT"
    ),
    (
        F.col("_transaction_type").isin(["CREDIT", "DEBIT"]) &
        F.col("_transaction_amount").isNotNull() &
        F.col("_signed_amount").isNotNull() &
        (
            F.abs(
                F.abs(F.col("_signed_amount")) -
                F.col("_transaction_amount")
            ) > 0.01
        ),
        "SIGNED_AMOUNT_MISMATCH"
    ),
    (
        (F.col("_transaction_type") == "CREDIT") &
        (F.col("_signed_amount") <= 0),
        "CREDIT_MUST_HAVE_POSITIVE_SIGN"
    ),
    (
        (F.col("_transaction_type") == "DEBIT") &
        (F.col("_signed_amount") >= 0),
        "DEBIT_MUST_HAVE_NEGATIVE_SIGN"
    ),

    # Balance after transaction
    (
        F.col("balance_after_txn").isNull() |
        F.col("_balance").isNull() |
        F.isnan(F.col("_balance")) |
        (F.abs(F.col("_balance")) == float("inf")) |
        (F.col("_balance") < 0),
        "INVALID_BALANCE_AFTER_TXN"
    ),

    # Currency
    (
        F.col("_currency").isNull() |
        (~F.col("_currency").isin(allowed_currencies)),
        "INVALID_CURRENCY"
    ),

    # Narration and category
    (
        F.col("narration").isNull() |
        (F.trim(F.col("narration")) == ""),
        "MISSING_NARRATION"
    ),
    (
        F.col("transaction_category").isNull() |
        (F.trim(F.col("transaction_category")) == ""),
        "MISSING_TRANSACTION_CATEGORY"
    ),

    # Salary-like flag
    (
        F.col("is_salary_like").isNull() |
        F.col("_salary_flag").isNull() |
        (~F.col("_salary_flag").isin(allowed_salary_flags)),
        "INVALID_SALARY_FLAG"
    ),

    # Source file reference
    (
        F.col("source_file_id").isNull() |
        (F.trim(F.col("source_file_id")) == ""),
        "MISSING_SOURCE_FILE_ID"
    ),

    # Transaction timestamp
    (
        F.col("transaction_ts").isNull() |
        F.col("_transaction_ts").isNull() |
        F.col("_dt_transaction_ts").isNull() |
        (F.col("_dt_transaction_ts") > F.current_timestamp()),
        "INVALID_TRANSACTION_TIMESTAMP"
    ),

    # Created and updated timestamps
    (
        F.col("created_at").isNull() |
        F.col("_created_at").isNull() |
        F.col("_dt_created_at").isNull(),
        "INVALID_CREATED_AT"
    ),
    (
        F.col("updated_at").isNull() |
        F.col("_updated_at").isNull() |
        F.col("_dt_updated_at").isNull(),
        "INVALID_UPDATED_AT"
    ),
    (
        F.col("_dt_created_at").isNotNull() &
        F.col("_dt_updated_at").isNotNull() &
        (F.col("_dt_updated_at") < F.col("_dt_created_at")),
        "UPDATED_BEFORE_CREATED"
    )
]

# ============================================================
# 6. CAPTURE ALL FAILURE REASONS
# ============================================================

df = df.withColumn(
    "dq_failed_reasons",
    F.concat_ws(
        " | ",
        *[
            F.when(condition, F.lit(reason))
            for condition, reason in rules
        ]
    )
)

# ============================================================
# 7. SPLIT PASSED AND QUARANTINED RECORDS
# ============================================================

helper_columns = [
    "_bank_txn_id", "_customer_id", "_account_id",
    "_transaction_ts", "_transaction_type",
    "_transaction_amount", "_signed_amount", "_balance",
    "_currency", "_bank_name", "_salary_flag",
    "_source_file_id", "_created_at", "_updated_at",
    "_dt_transaction_ts", "_dt_created_at", "_dt_updated_at",
    "_txn_id_count"
]

original_columns = source_df.columns

bank_data_passed = (
    df.filter(F.col("dq_failed_reasons") == "")
      .select(*original_columns)
)

quarantine_data = (
    df.filter(F.col("dq_failed_reasons") != "")
      .drop(*helper_columns)
)

# ============================================================
# 8. QUALITY SUMMARY
# ============================================================

source_count = source_df.count()
passed_count = bank_data_passed.count()
failed_count = quarantine_data.count()

print(f"Source records      : {source_count}")
print(f"Passed records      : {passed_count}")
print(f"Quarantined records : {failed_count}")
print(f"Reconciled records  : {passed_count + failed_count}")

# View quarantined records and failure reasons
display(
    quarantine_data.select(
        "bank_txn_id",
        "customer_id",
        "account_id",
        "transaction_type",
        "transaction_amount",
        "signed_amount",
        "balance_after_txn",
        "dq_failed_reasons"
    )
)

# View passed records
display(bank_data_passed)


In [0]:

# Create destination schemas if they do not exist
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.data_quality_check
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.quarantine_data
""")

# Append valid bank transactions
(bank_data_passed.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.data_quality_check.bank_data"))

# Append failed bank transactions with failure reasons
(quarantine_data.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.quarantine_data.bank_data"))

print("Bank Data Quality Gate completed.")
print("Passed data appended to fintech360.data_quality_check.bank_data")
print("Failed data appended to fintech360.quaratine_data.bank_data")
